In [25]:
import numpy as np 
import pandas as pd
import torch.nn as nn
import torch
import matplotlib.pyplot as plt
import sklearn
from sklearn.preprocessing import StandardScaler
import sys, os

root = os.path.abspath(os.path.join(os.getcwd(), '..',))

sys.path.insert(0, root)

from utils import time_series_split
from FFN import FCN

In [26]:
df = pd.read_csv('..\data\BTC_USDT_5m.csv', parse_dates=['open_time'], index_col=['open_time'])
print(df.head())
print(df.info())

                         open      high       low     close     volume
open_time                                                             
2017-12-31 00:00:00  12345.10  12397.16  12287.89  12311.02  36.479616
2017-12-31 00:05:00  12313.00  12410.46  12270.74  12270.74  56.512808
2017-12-31 00:10:00  12270.74  12300.12  12149.98  12211.95  75.285130
2017-12-31 00:15:00  12206.82  12489.05  12201.27  12391.00  70.083180
2017-12-31 00:20:00  12379.92  12489.04  12296.28  12468.41  87.112825
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 762998 entries, 2017-12-31 00:00:00 to 2025-04-07 21:20:00
Data columns (total 5 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   open    762998 non-null  float64
 1   high    762998 non-null  float64
 2   low     762998 non-null  float64
 3   close   762998 non-null  float64
 4   volume  762998 non-null  float64
dtypes: float64(5)
memory usage: 34.9 MB
None


In [27]:
df.isna().sum()

open      0
high      0
low       0
close     0
volume    0
dtype: int64

## Scaling Data

In [28]:
scalar_x = StandardScaler()
scalar_y = StandardScaler()

In [29]:
x = df[['open', 'high', 'low', 'volume']]
y = df.close


In [30]:
## inputs: converting numpy default float64(double  precision) to torch default(float32)

x = torch.from_numpy(np.array(x).astype(np.float32))
y = torch.from_numpy(np.array(y).astype(np.float32))


X_train, X_test, y_train, y_test = time_series_split(x, y, .80)

xtrain_scaled = scalar_x.fit_transform(X_train)
ytrain_scaled = scalar_y.fit_transform(y_train.reshape(-1, 1)).flatten()



X_train_scaled = torch.from_numpy(xtrain_scaled.astype(np.float32))
y_train_scaled = torch.from_numpy(ytrain_scaled.astype(np.float32))

## Data Splits

print(X_train_scaled.shape)
print(y_train_scaled.shape)



Times-series Split:
Training Samples: 610398 (first 80% of data) 
Testing Samples: 152600 (last 20% of data) 
torch.Size([610398, 4])
torch.Size([610398])


In [36]:
## Torch dataset
from torch.utils.data import TensorDataset, DataLoader
dataset = TensorDataset(X_train_scaled, y_train)

batch_size = 100
train_loader = DataLoader(dataset, batch_size, shuffle=False)
train_loader

### Linear Regression

In [37]:
## Parameters
learning_rate = 0.001
n_iters = 100
input_size = X_train.shape[1]
output_size = 1

print(input_size)

4


In [38]:
## model
reg = nn.Linear(input_size, output_size, dtype=torch.float32)

In [39]:
## Loss
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(reg.parameters(), learning_rate)


In [ ]:
## Training
for iter in range(n_iters+1):
    for batch_X, batch_y in train_loader:
          
        #Forward pass
        pred = reg(batch_X)
        pred.shape
        loss_value = criterion(pred, batch_y)

        #Backward pass & optimize
        optimizer.zero_grad()
        loss_value.backward()
        optimizer.step()


    if iter % 10 == 0:
        print(f'Epoch {iter}, Loss: {loss_value.item():.4f}')



c:\Users\user\Documents\dev\deeplearning\venv\lib\site-packages\torch\nn\modules\loss.py:626: UserWarning: Using a target size (torch.Size([100])) that is different to the input size (torch.Size([100, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
c:\Users\user\Documents\dev\deeplearning\venv\lib\site-packages\torch\nn\modules\loss.py:626: UserWarning: Using a target size (torch.Size([98])) that is different to the input size (torch.Size([98, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch 0, Loss: 1436041.5000
Epoch 10, Loss: 16658.2129
Epoch 20, Loss: 16657.8867
Epoch 30, Loss: 16657.6973
Epoch 40, Loss: 16657.2832


### FCN Model

In [ ]:
## Parameters
learning_rate = 0.001
n_iters = 100
input_size = X_train.shape[1]
output_size = 1

print(input_size)

In [ ]:
fcn = FCN([4, 12, 8, 1], activation='tanh')

In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(fcn.parameters(), learning_rate)


In [ ]:
#Training
for iter in range(n_iters+1):
    for batch_X, batch_y in train_loader:

        #forward pass
        pred = fcn.forward(batch_X)
        loss_value = criterion(pred, batch_y)

        #Backward pass & optimize
        optimizer.zero_grad()
        loss_value.backward()
        optimizer.step()

    if iter % 10 == 0:
        print(f'Epoch {iter}, Loss: {loss_value.item():.4f}')

        

### Testing


In [ ]:
X_test_scaled = torch.tensor(scalar_x.transform(X_test), dtype=torch.float32)
y_test_scaled = torch.tensor(scalar_y.transform(y_test.reshape(-1,1)).flatten(), dtype=torch.float32)
y_test_scaled

#### Regression model

In [ ]:
##Preditions 
reg.eval()
with torch.no_grad():
    predictions_scaled = reg(X_test_scaled).numpy()
    
predictions_scaled

# Inverse transform the scaled predictions
reg_predictions = scalar_y.inverse_transform(predictions_scaled.reshape(-1, 1)).flatten()
reg_predictions

In [ ]:
## Metric
from sklearn.metrics import mean_squared_error

MSE = mean_squared_error(y_test, reg_predictions)
print(f'Regression model MSE on test data: {MSE}')
print(f'Regression model RMSE on test data: {np.sqrt(MSE)}')

In [ ]:
# ## Graph

# # fig, axes = plt.subplots(1, 2, figsize=(20, 6))

# plt.plot(y_test[:100], 'g-', label='original', alpha=.7)
# plt.plot(predictions[:100], 'r-', label='predicted', alpha=.7, linewidth=.8)

# plt.legend()
# plt.show()

## Residual Plotting


In [ ]:
pred = predictions.flatten()
# y_test = y_test.numpy()
print(y_test)
print(pred)


In [ ]:
from scipy import stats
fig, axes = plt.subplots(2, 2, figsize=(15,12))

#Residual vs predictions
residuals = y_test - pred
axes[0,0].scatter(predictions, residuals, alpha=.5)
axes[0,0].set_xlabel('predictions')
axes[0,0].set_ylabel('residuals')
axes[0,0].set_title('Residual vs Predictions')
axes[0, 0].grid(True, alpha=.5)


# Residuals vs Target
axes[0,1].scatter(y_test, residuals, alpha=.5)
axes[0,1].set_xlabel('Targets')
axes[0,1].set_ylabel('residuals')
axes[0,1].set_title('residual vs Target')
axes[0,1].grid(True, alpha=.5)

## distribution of residuals
axes[1,0].hist(residuals, bins=50)
axes[1,0].set_xlabel('Residuals')
axes[1,0].set_ylabel('Frequency')
axes[1,0].set_title('Distribution of Residuals')
axes[1,0].grid(True, alpha=.5)

##Q-Q plot
stats.probplot(residuals, dist='norm', plot=axes[1,1])
axes[1,1].set_title('Q-Q plot')
axes[1,1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
from 